# Notebook 04: Full End-to-End Fine-Tuning

In this notebook, we perform true end-to-end training. **All weights of the Moirai encoder and the classification heads are updated**.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display

from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import  *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16] #[8, 16, 32, 64]

lr_grid=[5e-5]
wd_grid=[0.05]

BATCH_SIZE = 32 

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

## 1. Baseline: Linear Model on Mask Embedding Only (Full FT)

In [3]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = universal_grid_search(
        model_class=FullMaskOnlyWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid= wd_grid, lr_grid = lr_grid
    )
    df_mask_only.loc["Mask Only", patch] = test_acc

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.1578
Acc on Test Set : 0.6265



In [4]:
display(df_mask_only.astype(float).round(4))

Patch Size,16
Mask Only,0.6265


## 2. Baseline: Mean Pooling

In [5]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, test_acc = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid= wd_grid, lr_grid = lr_grid
    )
    df_mean_pool.loc["Mean Pooling", patch] = test_acc

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.1380
Acc on Test Set : 0.6399



In [6]:
display(df_mean_pool.astype(float).round(4))

Patch Size,16
Mean Pooling,0.6399


## 3. Advanced Pooling: Attention & MHA (Full FT)

In [7]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, acc_attn = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid= wd_grid, lr_grid = lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_attn

        # 2. Multi-Head Attention
        _, acc_mha = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
                wd_grid=wd_grid, lr_grid = lr_grid
        )
        df_adv_single.loc[(model_names_single[1], mode), patch] = acc_mha

display(df_adv_single.astype(float).round(4))

LR=5e-05 | WD=0.05
 Early stopping : 35
Val Loss: 1.0713
Acc on Test Set : 0.6504

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0275
Acc on Test Set : 0.6492

LR=5e-05 | WD=0.05
 Early stopping : 36
Val Loss: 1.1055
Acc on Test Set : 0.6350

LR=5e-05 | WD=0.05
 Early stopping : 35
Val Loss: 1.1134
Acc on Test Set : 0.6318

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.1571
Acc on Test Set : 0.6444

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.1112
Acc on Test Set : 0.6350

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.1057
Acc on Test Set : 0.6436

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.1514
Acc on Test Set : 0.6253



Patch Size                                8       16
Model            Mode                               
Attention (Base) shared_context       0.6504  0.6444
                 independent_context  0.6350  0.6436
MHA (Heads=16)   shared_context       0.6492  0.6350
                 independent_context  0.6318  0.6253

## 4. Ablation Study: Number of Attention Heads

In [ ]:
PATCH = 8
HEADS_TO_TEST = [16, 32, 64, 128] 

df_heads_ablation = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation.index.name = "Mode"
df_heads_ablation.columns.name = f"Num Heads (Patch {PATCH})"

for mode in MODES:
    for heads in HEADS_TO_TEST:
        _, test_acc = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_heads_ablation.loc[mode, heads] = test_acc

LR=5e-05 | WD=0.05
 Early stopping : 35
Val Loss: 1.0528
Acc on Test Set : 0.6399

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.0482
Acc on Test Set : 0.6492

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.0621
Acc on Test Set : 0.6488

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.0287
Acc on Test Set : 0.6363

LR=5e-05 | WD=0.05
 Early stopping : 36
Val Loss: 1.1364
Acc on Test Set : 0.6346

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0434
Acc on Test Set : 0.6480

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0650
Acc on Test Set : 0.6472

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0888
Acc on Test Set : 0.6354



Num Heads (Patch 8),16,32,64,128
Mode,,,,
shared_context,0.6399,0.6492,0.6488,0.6363
independent_context,0.6346,0.6480,0.6472,0.6354


In [9]:
display(df_heads_ablation.astype(float).round(4))

Num Heads (Patch 8),16,32,64,128
Mode,,,,
shared_context,0.6399,0.6492,0.6488,0.6363
independent_context,0.6346,0.6480,0.6472,0.6354


In [10]:
PATCH = 16
HEADS_TO_TEST = [16, 32, 64, 128] 

df_heads_ablation = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation.index.name = "Mode"
df_heads_ablation.columns.name = f"Num Heads (Patch {PATCH})"

for mode in MODES:
    for heads in HEADS_TO_TEST:
        _, test_acc = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_heads_ablation.loc[mode, heads] = test_acc

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0549
Acc on Test Set : 0.6269

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.1071
Acc on Test Set : 0.6294

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.1012
Acc on Test Set : 0.6273

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0962
Acc on Test Set : 0.6269

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.0919
Acc on Test Set : 0.6436

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.0718
Acc on Test Set : 0.6111

LR=5e-05 | WD=0.05
 Early stopping : 33
Val Loss: 1.1141
Acc on Test Set : 0.6261

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.0989
Acc on Test Set : 0.6257



In [11]:
display(df_heads_ablation.astype(float).round(4))

Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.6269,0.6294,0.6273,0.6269
independent_context,0.6436,0.6111,0.6261,0.6257


## 5. Hierarchical MHA (Full FT)

In [12]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_hier = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = acc_hier

LR=5e-05 | WD=0.05


 Early stopping : 32
Val Loss: 1.2419
Acc on Test Set : 0.5783

LR=5e-05 | WD=0.05
 Early stopping : 34
Val Loss: 1.1956
Acc on Test Set : 0.5969

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.2525
Acc on Test Set : 0.5839

LR=5e-05 | WD=0.05
 Early stopping : 32
Val Loss: 1.2836
Acc on Test Set : 0.5795



In [13]:
display(df_hierarchical.astype(float).round(4))

Patch Size,8,16
Mode,,
shared_context,0.5783,0.5839
independent_context,0.5969,0.5795
